# Models trained on data after Balancing and Sampling

In [3]:
import sys
print(f"Python version: {sys.version}")
import json
import logging
import csv
import gzip
import re
import pandas as pd
import numpy as np
from functools import reduce
from pyspark.sql.types import StringType,DecimalType,DoubleType,IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler, Bucketizer
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession,Row
from pyspark import SparkConf
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# -----------------------------------------------------------------------------
# INITIALIZE LOGGING
# -----------------------------------------------------------------------------
f = '%(asctime)-15s %(levelname)-8s %(message)s'
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")
logging.basicConfig(format=f)


from IPython.core.magic import register_cell_magic

# -----------------------------------------------------------------------------
# start_spark
# -----------------------------------------------------------------------------
def start_spark(
    driver_memory="100g",
    storage_fraction=0.5,
    num_nodes=10,
):
    """Initialize spark

    Arguments:
        driver_memory: Maximum heap size for the Spark driver Java
            virtual machine.
        storage_fraction: Controls what portion of Spark's unified
            memory is reserved for storage (i.e., caching/persisting data
            and broadcast variables), as a fraction of the total
            execution + storage memory pool.
            If you cache/persist a lot of data, and you're evicting
            data too early, you might increase this value (e.g. 0.6 or 0.7).
            Conversely, if your job is shuffle-heavy and fails due to
            memory pressure, you might decrease it (e.g. 0.3).
        num_nodes: How many concurrent threads to use while running
            in "local mode" (i.e. in a single machine instead of a cluster).
            Use '*' to use all cores, or an integer > 0 for a specific
            number of threads.
    """

    conf = SparkConf().setAppName("My_Application")
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.memory.storageFraction", str(storage_fraction))
    conf.setMaster(f"local[{num_nodes}]")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel('WARN')

    return spark


Python version: 3.11.0 (main, Jun 13 2025, 14:48:45) [Clang 16.0.0 (clang-1600.0.26.6)]


In [4]:

spark = start_spark(num_nodes=10)
#spark.stop()

  
@register_cell_magic
def spark_sql(line, cell):
    result = spark.sql(cell)
    result.show(n=1000)
  

# -- READ ENROLLMENT AND DATA TABLES
enrollment_file = f"/Users/Charles/DATA/ckd/ckd_enrollment"
logger.info(f">>> Reading enrollment file: {enrollment_file}")
df_enrollment = spark.read.format("parquet").load(enrollment_file)
df_enrollment.createOrReplaceTempView('enrollment')
logger.info(f">>> ENROLLMENT has {df_enrollment.count():,} rows")
logger.info(f">>> ENROLLMENT has {df_enrollment.select('ENROLID').distinct().count():,} unique enrollees")

claims_file = f"/Users/Charles/DATA/ckd/ckd_claims"
logger.info(f">>> Reading claims file: {claims_file}")
df_claims = spark.read.format("parquet").load(claims_file)
df_claims.createOrReplaceTempView('claims')
logger.info(f">>> CLAIMS has {df_claims.count():,} rows")
logger.info(f">>> CLAIMS has {df_claims.select('ENROLID').distinct().count():,} unique enrollees")

# -- NOTE: with the "createOrReplaceTempView" we define a view of these
# -- tables, so we can use them in SQL queries.

2025-09-11 15:15:21,379 INFO     >>> Reading enrollment file: /Users/Charles/DATA/ckd/ckd_enrollment


2025-09-11 15:15:21,534 INFO     >>> ENROLLMENT has 2,586,930 rows
2025-09-11 15:15:21,990 INFO     >>> ENROLLMENT has 862,310 unique enrollees
2025-09-11 15:15:21,991 INFO     >>> Reading claims file: /Users/Charles/DATA/ckd/ckd_claims
2025-09-11 15:15:22,353 INFO     >>> CLAIMS has 253,070,875 rows
2025-09-11 15:15:29,359 INFO     >>> CLAIMS has 862,310 unique enrollees        


# Load Data

In [7]:
train_df = pd.read_csv("train_df_logisticAT_0911.csv")
test_df = pd.read_csv("test_df_logisticAT_0911.csv")
print(train_df.shape, test_df.shape)

(23485, 91) (10066, 91)


In [8]:
import importlib
import model_pipeline
importlib.reload(model_pipeline)

# Correctly import your custom modules
import evaluate_model_by_cost_stratum
import evaluate_model_by_risk_bin

# Then reload them
importlib.reload(evaluate_model_by_risk_bin)
importlib.reload(evaluate_model_by_cost_stratum)

<module 'evaluate_model_by_cost_stratum' from '/Users/cat2510/my_projects/evaluate_model_by_cost_stratum.py'>

# Feature groups, define high cost and train-test-split

In [11]:
BIN_FLAG_COLUMNS = model_pipeline.get_bin_flag_columns(train_df) +['lab_monitoring_adherent','nephrology_consult_adherent','early_nephrology_referral']
STAGE_COLUMNS = ['2017Q1', '2017Q2', '2017Q3', '2017Q4',"stage_2017",'2017Q1_max_ckd_stage','2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage','2017Q4_max_ckd_stage', ]
CAT_COLUMNS = train_df.select_dtypes(include=["object","category"]).columns.tolist() + ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION']
TRUE_NUM_COLUMNS = model_pipeline.get_true_num_columns(train_df,CAT_COLUMNS)+[ 'util_2017', 'total_increasing_quarters_2017'
, 'total_lab_tests', ]
print(CAT_COLUMNS)

leftover_cols = [
    c for c in train_df.columns 
    if c not in CAT_COLUMNS and c not in TRUE_NUM_COLUMNS and c not in STAGE_COLUMNS and c not in BIN_FLAG_COLUMNS 
]

print(f"Number of leftover columns: {len(leftover_cols)}")
print(leftover_cols, train_df.shape)  # preview first 50
print(train_df.shape, test_df.shape)

['cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity', 'risk_bin', 'stage_group', 'INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION']
Number of leftover columns: 3
['ENROLID', 'nephrology_visit_count', 'true_class'] (23485, 91)
(23485, 91) (10066, 91)


In [12]:
from numpy.char import startswith


feature_cols = [c for c in train_df.columns
              if c not in  ('risk_score','risk_bin',"ENROLID",'nephrology_visit_count', 'true_class',"cost_stratum_2018", "high_cost_2018",'annual_cost17')
              and not c.startswith("highcost_gt_")]    # keep only predictors
feature_cols


['INCOME_LEVEL',
 'AGEGRP',
 'SEX',
 'REGION',
 'has_Hypertension',
 'has_Type_2_Diabetes',
 'has_Anemia',
 'has_Hyperlipidemia',
 'has_Acute_Kidney_Failure',
 'has_Hyperparathyroidism',
 'has_Kidney_Transplant',
 'has_Vitamin_D_Deficiency',
 'has_Long-term_Drug_Therapy',
 'has_Hypothyroidism',
 'has_Sleep_Apnea',
 'stage_2017',
 'util_2017',
 'THRCLS_53',
 'THRCLS_51',
 'THRCLS_52',
 'THRCLS_69',
 'THRCLS_46',
 'THRCLS_47',
 'THRCLS_172',
 'THRCLS_174',
 'THRCLS_181',
 'THRCLS_60',
 '2017Q1_ckd_cost',
 '2017Q1_ckd_claims',
 '2017Q1_max_ckd_stage',
 '2017Q1_direct_ckd_cost',
 '2017Q1_procedure_ckd_cost',
 '2017Q1_comorbidity_ckd_cost',
 '2017Q2_ckd_cost',
 '2017Q2_ckd_claims',
 '2017Q2_max_ckd_stage',
 '2017Q2_direct_ckd_cost',
 '2017Q2_procedure_ckd_cost',
 '2017Q2_comorbidity_ckd_cost',
 '2017Q3_ckd_cost',
 '2017Q3_ckd_claims',
 '2017Q3_max_ckd_stage',
 '2017Q3_direct_ckd_cost',
 '2017Q3_procedure_ckd_cost',
 '2017Q3_comorbidity_ckd_cost',
 '2017Q4_ckd_cost',
 '2017Q4_ckd_claims',
 '

In [14]:
import balancing_functions.optimal_match_control
importlib.reload(balancing_functions.optimal_match_control)
from balancing_functions.optimal_match_control import *

# Matching

In [15]:
importlib.reload(balancing_functions.optimal_match_control)
from balancing_functions.optimal_match_control import *

# Create your resampler with OR-Tools
sampler = EnhancedRiskBinnedCaseControlResampler(
    matching_method="ortools",  # Use OR-Tools for optimization
    random_state=42,
    proba_col="risk_score",  # Your probability column
    binary_group="true_class",       # Your binary target column
    uid_col="ENROLID"            # Your ID column
)

# Get bin strategies optimized for OR-Tools
strategies = get_high_cost_detection_config(primary_matching_method="ortools")

# Calculate the size of the smallest bin before matching
bin_counts = train_df["risk_bin"].value_counts()
smallest_bin_size = bin_counts.min()
print(f"Smallest bin size before matching: {smallest_bin_size}")
print(f"Bin sizes before matching: {bin_counts.to_dict()}")

# Apply matching with equalization to the smallest bin size
matched_train_df, removed_ids = sampler.apply_sampler_by_bin(
    train_df,
    bin_specific_strategies=strategies,
    target_bin_size=smallest_bin_size,  # Target the smallest bin size
    verbose=True
)



Smallest bin size before matching: 343
Bin sizes before matching: {'Stratum_1': 6235, 'Stratum_2': 6005, 'Stratum_3': 4643, 'Stratum_4': 2964, 'Stratum_5': 1590, 'Stratum_0': 1037, 'Stratum_6': 668, 'Stratum_7': 343}
Target bin size: 343
→ We will not match on: ['ENROLID', 'true_class', 'risk_score']

--- Processing bin Stratum_0 (n=1037) ---
Using matching method:  ortools
=== ENTERING ortools_optimization_matching ===
>>> Distance computation will use 88 features
    Categorical features (one-hot encoded): ['cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity', 'stage_group']
    Numeric   features (normalized): ['INCOME_LEVEL', 'annual_cost17', 'highcost_gt_50000', 'highcost_gt_75000', 'highcost_gt_100000', 'highcost_gt_200000', 'highcost_gt_300000', 'highcost_gt_400000', 'highcost_gt_500000', 'AGEGRP', 'SEX', 'REGION', 'stage_2017', 'util_2017', '2017Q1_ckd_cost', '2017Q1_ckd_claims', '2017Q1_max_ckd_stage', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2

In [16]:
# Verify the results
print("\nAfter matching with equalization:")
print(f"Total samples: {len(matched_train_df)}")
print(f"Cases: {(matched_train_df['true_class'] == 1).sum()}")
print(f"Controls: {(matched_train_df['true_class'] == 0).sum()}")
print(f"Bin sizes after matching: {matched_train_df['risk_bin'].value_counts().to_dict()}")


After matching with equalization:
Total samples: 4717
Cases: 2385
Controls: 2332
Bin sizes after matching: {'Stratum_5': 994, 'Stratum_1': 686, 'Stratum_2': 686, 'Stratum_3': 686, 'Stratum_4': 686, 'Stratum_6': 534, 'Stratum_7': 343, 'Stratum_0': 102}


In [17]:
bins = [f"{x:.1f}-{x+0.1:.1f}" for x in np.arange(0.0, 1.0, 0.1)]

fig = px.histogram(
    matched_train_df,
    x="risk_bin",
    color="true_class",
    category_orders={"risk_bin": bins},   # good to keep
    labels={
        "risk_bin": f"Predicted Risk Score",
        "true_class": "Stage ≤3 vs >3"
    },
    title="Matched Cohort: High-Cost Risk by 2017 CKD Stage Group",
    barmode="group",
    height=500, width=800
)

# *** pin the x-axis order ***
fig.update_xaxes(categoryorder="array", categoryarray=bins)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [18]:
# 1) Collapse to one row per ENROLID
matched_ids_df = (
    matched_train_df[["ENROLID"]]
      .drop_duplicates()
      .reset_index(drop=True)
)

# 2) Pull in the target from your original train set
matched_with_target = matched_ids_df.merge(
    train_df[["ENROLID", "high_cost_2018"]],
    on="ENROLID",
    how="inner"
)
print(matched_with_target.shape)
print(CAT_COLUMNS)


(4717, 2)
['cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity', 'risk_bin', 'stage_group', 'INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION']


# Model Comparisons

In [19]:
import importlib
import model_pipeline
importlib.reload(model_pipeline)

# Import with correct syntax
import evaluate_model_by_cost_stratum
import evaluate_model_by_risk_bin

# Reload
importlib.reload(evaluate_model_by_cost_stratum)
importlib.reload(evaluate_model_by_risk_bin)
from evaluate_model_by_risk_bin import evaluate_metrics_by_risk_bin, plot_probability_and_calibration


In [20]:
X_train, y_train = matched_train_df[feature_cols], matched_with_target['high_cost_2018']
X_test,y_test = test_df[feature_cols], test_df['high_cost_2018']
# 5. Create and train models using your existing pipelines
# Logistic Regression model
log_pipeline_matched = model_pipeline.get_logistic_pipeline(
    df=matched_train_df[feature_cols]
,categorical_cols=CAT_COLUMNS, numeric_cols=TRUE_NUM_COLUMNS
)
log_pipeline_matched.fit(X_train, y_train)

# Random Forest model
rf_pipeline_matched = model_pipeline.get_random_forest_pipeline(
    df=matched_train_df[feature_cols]
,categorical_cols=CAT_COLUMNS, numeric_cols=TRUE_NUM_COLUMNS
)
rf_pipeline_matched.fit(X_train, y_train)

# Gradient Boosting model
gb_pipeline_matched = model_pipeline.get_histgb_pipeline(
    df=matched_train_df[feature_cols]
,categorical_cols=CAT_COLUMNS, numeric_cols=TRUE_NUM_COLUMNS
)
gb_pipeline_matched.fit(X_train,y_train)


→ Building preprocessor:
   • OneHotEncoder on: ['cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity', 'stage_group', 'INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION']
   • StandardScaler on: ['2017Q1_ckd_cost', '2017Q1_ckd_claims', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_ckd_claims', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_ckd_claims', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4_ckd_claims', '2017Q4_direct_ckd_cost', '2017Q4_procedure_ckd_cost', '2017Q4_comorbidity_ckd_cost', 'ckd_cost_trend_2017', 'ckd_cost_volatility_2017', 'ckd_cost_deriv_Q1_Q2_2017', 'ckd_cost_deriv_Q2_Q3_2017', 'ckd_cost_deriv_Q3_Q4_2017', 'avg_quarterly_derivative_2017', 'quarterly_std_2017', 'quarterly_skewness_2017', 'quarterly_kurtosis_2017', 'quarterly_cv_2017', 'quarterly_max_2017', 'q

/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning:


The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).




Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['cost_pattern_2017',
                                                   'cost_stability_2017',
                                                   'lab_monitoring_intensity',
                                                   'stage_group',
                                                   'INCOME_LEVEL', 'AGEGRP',
                                                   'SEX', 'REGION']),
                                                 ('num', StandardScaler(),
                                                  ['2017Q1_ckd_cost',
                                                   '2017Q1_ckd_claims',
                                                   '2017Q1_direct_...
                                                   'ckd_cost_trend_2017',
                                                   'ckd_cost_volatility_2017',
                                                   'ckd_cost_deriv_Q1_Q2_2017',
                                                   'ckd_cost_deriv_Q2_Q3_2017',
                                                   'ckd_cost_deriv_Q3_Q4_2017',
                                                   'avg_quarterly_derivative_2017',
                                                   'quarterly_std_2017',
                                                   'quarterly_skewness_2017',
                                                   'quarterly_kurtosis_2017',
                                                   'quarterly_cv_2017', ...])])),
                ('classifier',
                 HistGradientBoostingClassifier(class_weight='balanced',
                                                max_iter=200,
                                                random_state=42))])

In [21]:


log_results_after = evaluate_metrics_by_risk_bin(
    log_pipeline_matched, X_test, y_test, test_df['risk_bin'],
    model_name='Logistic Regression',
    dataset_name='After Matching'
)


# Random Forest results AFTER matching
rf_results_after = evaluate_metrics_by_risk_bin(
    rf_pipeline_matched, X_test, y_test, test_df['risk_bin'],
    model_name='Random Forest',
    dataset_name='After Matching'
)


# Gradient Boosting results AFTER matching
gb_results_after = evaluate_metrics_by_risk_bin(
    gb_pipeline_matched, X_test, y_test, test_df['risk_bin'],
    model_name='Balanced XGBoost',
    dataset_name='After Matching'
)


# Concatenate all results
all_results = pd.concat([
    log_results_after, 
   gb_results_after,
    rf_results_after, 
], ignore_index=True)

# Compare models
evaluate_model_by_risk_bin.plot_model_comparison_by_risk_bin(all_results, metric= 'recall')



KeyError: 'lower_bound'

<Figure size 1200x500 with 0 Axes>